# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

In [ ]:
"""
Books to Scrape - Web Scraping Script
======================================
Scrapes book data from https://books.toscrape.com/
and returns a filtered pandas DataFrame.

Usage
-----
    from scrape_books import scrape_books
    df = scrape_books(min_rating=4, max_price=20)
    print(df)

Or run directly:
    python scrape_books.py

Set DEMO_MODE = True (bottom of file) to run offline with synthetic data.
"""

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# ── Configuration ─────────────────────────────────────────────────────────────

BASE_URL = "https://books.toscrape.com/"

# Word-to-number rating map used on the website
RATING_MAP = {
    "One":   1,
    "Two":   2,
    "Three": 3,
    "Four":  4,
    "Five":  5,
}


# detail

def get_book_detail(detail_url: str) -> dict:
    """
    Fetch UPC, description, availability, and genre from a book's detail page.
    Returns a dict with keys: upc, availability, description, genre.
    """
    response = requests.get(detail_url, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    # UPC — inside the product information table
    upc = ""
    table = soup.find("table", class_="table-striped")
    if table:
        for row in table.find_all("tr"):
            th = row.find("th")
            td = row.find("td")
            if th and td and th.text.strip() == "UPC":
                upc = td.text.strip()
                break

    # Availability
    availability = ""
    avail_p = soup.find("p", class_="instock availability")
    if avail_p:
        availability = avail_p.text.strip()

    # Description — paragraph right after the #product_description div
    description = ""
    desc_div = soup.find("div", id="product_description")
    if desc_div:
        desc_p = desc_div.find_next_sibling("p")
        if desc_p:
            description = desc_p.text.strip()

    # Genre — 3rd breadcrumb: Home > Books > [Genre] > Title
    genre = ""
    breadcrumb = soup.find("ul", class_="breadcrumb")
    if breadcrumb:
        crumbs = breadcrumb.find_all("li")
        if len(crumbs) >= 3:
            genre = crumbs[2].text.strip()

    return {
        "upc":          upc,
        "availability": availability,
        "description":  description,
        "genre":        genre,
    }


# main scrap

def scrape_books(min_rating: float = 4.0, max_price: float = 20.0) -> pd.DataFrame:
    """
    Scrape Books to Scrape and return a filtered DataFrame.

    Parameters
    ----------
    min_rating : float
        Minimum star rating (1-5). Books with rating >= min_rating are kept.
    max_price  : float
        Maximum price in GBP. Books priced <= max_price are kept.

    Returns
    -------
    pd.DataFrame with columns:
        UPC | Title | Price (£) | Rating | Genre | Availability | Description
    """
    books = []
    page  = 1

    print(f"Scraping: rating >= {min_rating} stars  |  price <= £{max_price:.2f}")
    print("-" * 60)

    while True:
        url = BASE_URL if page == 1 else f"{BASE_URL}catalogue/page-{page}.html"

        response = requests.get(url, timeout=10)
        if response.status_code == 404:
            break
        response.raise_for_status()

        soup     = BeautifulSoup(response.text, "html.parser")
        articles = soup.find_all("article", class_="product_pod")
        if not articles:
            break

        for article in articles:

            # Rating
            rating_tag  = article.find("p", class_="star-rating")
            rating_word = rating_tag["class"][1] if rating_tag else "One"
            rating      = RATING_MAP.get(rating_word, 0)

            # Price
            price_tag  = article.find("p", class_="price_color")
            price_text = price_tag.text.strip() if price_tag else "£0"
            price_clean = price_text.replace("Â", "").replace("£", "").strip()
            try:
                price = float(price_clean)
            except ValueError:
                price = 0.0

            # Early filter — avoid unnecessary detail-page requests
            if rating < min_rating or price > max_price:
                continue

            # Title
            title_tag = article.find("h3").find("a")
            title     = title_tag["title"] if title_tag else "Unknown"

            # Detail URL
            relative_href = title_tag["href"] if title_tag else ""
            if relative_href.startswith("../"):
                relative_href = relative_href.replace("../", "")
            detail_url = BASE_URL + "catalogue/" + relative_href

            # Fetch detail page
            try:
                detail = get_book_detail(detail_url)
                time.sleep(0.1)   # Polite crawl delay
            except Exception as exc:
                print(f"  Warning: could not fetch detail for '{title}': {exc}")
                detail = {"upc": "", "availability": "", "description": "", "genre": ""}

            books.append({
                "UPC":          detail["upc"],
                "Title":        title,
                "Price (£)":    price,
                "Rating":       rating,
                "Genre":        detail["genre"],
                "Availability": detail["availability"],
                "Description":  detail["description"],
            })

        print(f"  Page {page:>3} | qualifying books so far: {len(books)}")
        page += 1

    df = pd.DataFrame(
        books,
        columns=["UPC", "Title", "Price (£)", "Rating", "Genre", "Availability", "Description"]
    )
    df = df.sort_values(["Rating", "Price (£)"], ascending=[False, True]).reset_index(drop=True)
    print("-" * 60)
    print(f"Done!  Total qualifying books: {len(df)}")
    return df


# ── Demo / offline mode ───────────────────────────────────────────────────────

def scrape_books_demo(min_rating: float = 4.0, max_price: float = 20.0) -> pd.DataFrame:
    """
    Offline demo that returns synthetic data matching the real scraper's schema.
    Useful for testing logic without an internet connection.
    """
    sample = [
        ("fake001abc123456", "The Great Alone",                  14.99, 5, "Fiction",           "In stock", "A gripping wilderness survival story."),
        ("fake002abc123456", "Where the Crawdads Sing",          12.50, 5, "Mystery",            "In stock", "A haunting story of isolation and murder."),
        ("fake003abc123456", "Educated",                          9.99, 5, "Biography",          "In stock", "A memoir about self-invention."),
        ("fake004abc123456", "Becoming",                         15.00, 5, "Biography",          "In stock", "Michelle Obama's memoir."),
        ("fake005abc123456", "The Midnight Library",             18.75, 4, "Fiction",            "In stock", "A library between life and death."),
        ("fake006abc123456", "Project Hail Mary",                19.99, 5, "Science Fiction",    "In stock", "A lone astronaut must save Earth."),
        ("fake007abc123456", "Atomic Habits",                    16.50, 4, "Self Help",          "In stock", "Tiny changes, remarkable results."),
        ("fake008abc123456", "The Seven Husbands of Evelyn Hugo",11.99, 4, "Historical Fiction", "In stock", "A reclusive star's confessions."),
        ("fake009abc123456", "Pachinko",                         17.00, 5, "Historical Fiction", "In stock", "Multi-generational saga of a Korean family."),
        ("fake010abc123456", "Anxious People",                   13.45, 4, "Fiction",            "In stock", "A funny, touching story about a hostage situation."),
        ("fake011abc123456", "Sapiens",                          25.00, 5, "History",            "In stock", "A brief history of humankind."),   # over £20
        ("fake012abc123456", "Sharp Objects",                    47.82, 4, "Mystery",            "In stock", "A dark psychological thriller."),  # over £20
        ("fake013abc123456", "The Boys in the Boat",             22.60, 4, "Sport",              "In stock", "Rowing triumph at the 1936 Olympics."),  # over £20
        ("fake014abc123456", "Shakespeare's Sonnets",            20.66, 4, "Poetry",             "In stock", "The complete collection of sonnets."),  # over £20
        ("fake015abc123456", "Set Me Free",                      17.46, 5, "Young Adult",        "In stock", "A coming-of-age novel."),
        ("fake016abc123456", "The Dirty Little Secrets",         33.34, 4, "Mystery",            "In stock", "Uncovering dark corporate secrets."),  # over £20
        ("fake017abc123456", "Starving Hearts",                  13.99, 4, "Romance",            "In stock", "A heartfelt contemporary romance."),
        ("fake018abc123456", "The Coming Woman",                 17.93, 3, "Historical Fiction", "In stock", "A story about 19th century women."),  # rating too low
        ("fake019abc123456", "A Light in the Attic",             19.50, 4, "Poetry",             "In stock", "A whimsical collection of poems."),
        ("fake020abc123456", "The Alchemist",                    10.00, 5, "Fiction",            "In stock", "A philosophical novel about destiny."),
    ]

    rows = [
        {"UPC": upc, "Title": title, "Price (£)": price, "Rating": rating,
         "Genre": genre, "Availability": avail, "Description": desc}
        for upc, title, price, rating, genre, avail, desc in sample
        if rating >= min_rating and price <= max_price
    ]

    df = pd.DataFrame(rows, columns=["UPC","Title","Price (£)","Rating","Genre","Availability","Description"])
    df = df.sort_values(["Rating","Price (£)"], ascending=[False, True]).reset_index(drop=True)
    print(f"[DEMO] {len(df)} books found (min_rating={min_rating}, max_price=£{max_price})")
    return df


# ── Entry point ───────────────────────────────────────────────────────────────

if __name__ == "__main__":
    import sys

    # Use demo mode when no internet is available (e.g. CI / sandbox).
    # Change to False (or remove the flag) to scrape the live website.
    USE_DEMO = "--live" not in sys.argv

    if USE_DEMO:
        print("Running in DEMO MODE (synthetic data). Pass --live to scrape the real site.\n")
        df = scrape_books_demo(min_rating=4, max_price=20)
    else:
        df = scrape_books(min_rating=4, max_price=20)

    # ── Display results ───────────────────────────────────────────────────
    pd.set_option("display.max_colwidth", 45)
    pd.set_option("display.width", 120)

    print("\n── Sample Output (first 10 rows) ──────────────────────────")
    print(df[["Title", "Price (£)", "Rating", "Genre"]].head(10).to_string(index=False))

    print("\n── Genre breakdown ────────────────────────────────────────")
    print(df["Genre"].value_counts().to_string())

    print("\n── Rating distribution ────────────────────────────────────")
    print(df["Rating"].value_counts().sort_index(ascending=False).to_string())

    print(f"\n── Price range : £{df['Price (£)'].min():.2f}  –  £{df['Price (£)'].max():.2f}")
    print(f"   Average price: £{df['Price (£)'].mean():.2f}")

    # Save to CSV
    output_csv = "books_scraped.csv"
    df.to_csv(output_csv, index=False)
    print(f"\nData saved to: {output_csv}")